Boruta（表情&音声）

In [37]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
import re
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from boruta import BorutaPy

pd.options.display.max_rows = None
#1.------前処理------

#表情データの読み込みとdfに格納
folder_path = Path('../../../output_filtered_questions')


file_list = folder_path.glob('*.csv')

file_path_list = [str(p) for p in file_list]

df = pd.DataFrame(file_path_list, columns=['filepath'])

files_df = df
files_df['filename'] = df['filepath'].apply(lambda p: Path(p).name)
files_df['ID'] = files_df['filename'].str.extract(r'ID(\d+)')
files_df['ID'] = files_df['ID'].astype(int)

#患者の疾患有無
labels_df = pd.read_csv(r"C:\Users\robotics\proj\Research\code\voice\delirium.csv")
#患者と疾患有無を結合
file_with_labels_df = pd.merge(files_df, labels_df, on='ID', how='left')
file_with_labels_df.drop(columns=['Sex'], inplace=True)

#print(file_with_labels_df)

#学習・テストデータとなるファイルの分割
X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
#print(X_files)
y_labels = file_with_labels_df.drop(columns=['filepath', 'filename', 'ID'], errors='ignore')
#rint(y_labels)

#層化５分割交差検証
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(X_files, y_labels)):
    X_train_sort, X_test_sort = X_files.iloc[train_index], X_files.iloc[test_index]
    y_train_sort, y_test_sort = y_labels.iloc[train_index], y_labels.iloc[test_index]

    #2.------学習データを読み込み、質問ごとの最大値を計算------

    train_file_path = [f for f in X_train_sort['filepath']]
    #print(train_file_path)
    train_df = pd.DataFrame()
    for f, a in zip(train_file_path, y_train_sort['Delirium']):
        df_train_file = pd.read_csv(f)
        #print(df_train_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #ファイル名の数字部分を抽出
        number = re.findall(r'\d+', current_filename)
        #print(number)
        df_train_file['Delirium'] = a
        df_train_file['filenumber'] = number[0]
        #print(df_train_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_train_file['label'] = df_train_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_train_sort_df = df_train_file[df_train_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_train_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_train_sort_df[confidence].index
        df_train_new_df = df_train_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_train_new_df.columns if col.startswith("AU")] + ["Delirium", "filenumber"]
        df_train_final_df = df_train_new_df[columns_to_extract]

        #print(df_train_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_train_final_df = df_train_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_train_final_df)

        #数値型に変更
        #df_train_final_df['filename'] = pd.to_numeric(df_train_final_df['filename'], errors='coerce')
        #print(df_train_final_df)

        #質問ごとの最大値を計算
        df_train_max_df = df_train_final_df.groupby('label').max()
        df_train_max_df = df_train_max_df.reset_index(drop=True)
        #print(df_train_max_df)

        train_df = pd.concat([train_df, df_train_max_df], ignore_index=True)

    #print(train_df)

    #3.------テストデータを読み込み、質問ごとの最大値を計算------

    test_file_path = [f for f in X_test_sort['filepath']]
    #print(test_file_path)
    test_df = pd.DataFrame()
    test_file_lengths = []
    for f, a in zip(test_file_path, y_test_sort['Delirium']):
        df_test_file = pd.read_csv(f)
        #print(df_test_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #print(current_filename)
        df_test_file['Delirium'] = a
        df_test_file['filename'] = current_filename
        #print(df_test_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_test_file['label'] = df_test_file['label'].astype(str).str.strip()
        
        #応答部分のAUだけを抽出
        df_test_sort_df = df_test_file[df_test_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_test_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_test_sort_df[confidence].index
        df_test_new_df = df_test_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_test_new_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_test_final_df = df_test_new_df[columns_to_extract]

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]

        #AU07, 11, 20の削除
        df_test_final_df = df_test_final_df.drop(columns=columns_to_AU, errors='ignore')
        #print(df_test_final_df)

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの最大値を計算
        df_test_max_df = df_test_final_df.groupby('label').max()
        df_test_max_df = df_test_max_df.reset_index(drop=True)
        #print(df_test_max_df)
        
        test_file_lengths.append(len(df_test_max_df))

        test_df = pd.concat([test_df, df_test_max_df], ignore_index=True)

    #print(test_df)
    #print(test_file_lengths)

    #4.------グリッドサーチのための学習・検証データの用意とテストデータの用意------

    y_train_grid = train_df['Delirium']
    y_test_grid = test_df['Delirium']

    columns_to_delete = ['Delirium']
    columns_to_delete2 = ['Delirium', 'filename']

    X_train_grid = train_df.drop(columns=columns_to_delete, errors='ignore')
    X_test_grid = test_df.drop(columns=columns_to_delete2, errors='ignore')

    #print(X_train_boruta)

"""     #患者ごとに分割するための変数を用意し、学習データからfilenumberを削除
    patient_groups = X_train_grid['filenumber'].values
    X_train_boruta = X_train_grid.drop(columns=['filenumber']).copy()

    #Add.------学習データだけを使用してBorutaを実行------

    #Borutaの実行
    Rf_boruta = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5)
    feat_selector = BorutaPy(Rf_boruta, n_estimators='auto', verbose=2, random_state=1)
    feat_selector.fit(X_train_boruta, y_train_grid)
    X_train_model = X_train_boruta.loc[:, feat_selector.support_]
    X_test_model = X_test_grid.loc[:, feat_selector.support_]

    #Borutaで採用、削除した特徴量の確認
    feature_names = np.array(X_train_boruta.columns)
    dropped_features = feature_names[~feat_selector.support_]
    accepted_features = feature_names[feat_selector.support_]
    tentative_features = feature_names[feat_selector.support_weak_]

    print(f"削除前の特徴量数：{len(feature_names)}")
    print(f"削除された特徴量数：{len(dropped_features)}")
    print("削除された特徴量一覧")
    print(dropped_features)

    #5.------学習の準備(学習器と学習方法の設定)------

    #モデル・グリッドサーチの定義
    model = RandomForestClassifier(random_state=42)
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10]
        }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=cv, n_jobs=-1)

    #6.------グリッドサーチ------
    grid_search.fit(X_train_model, y_train_grid, groups=patient_groups)
    print(f"---fold{fold+1}---\n")
    print(f"最適なハイパーパラメータ：{grid_search.best_params_}")
    print(f"CVでのmax精度(Accuracy):{grid_search.best_score_:.4f}")

    #行と列の表示制限を解除（または十分に大きな値に設定）
    pd.set_option('display.max_rows', None)     # すべての行を表示
    pd.set_option('display.max_columns', None)  # すべての列を表示
    pd.set_option('display.width', 1000)        # 表示幅を広くする

    #結果をDataFrameに変換し表示
    results_df = pd.DataFrame(grid_search.cv_results_)

    print("--- グリッドサーチの全試行結果 ---")
    print(results_df)

    #表示設定を元に戻す (推奨)
    #グローバルな設定を元に戻し、他の処理に影響を与えないようにします
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')


    #7.------テストデータでの性能評価------
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_model)
    final_accuracy = accuracy_score(y_test_grid, y_pred)

    print(f"テストデータでの正解率: {final_accuracy:.4}\n")
    #print(y_pred)

    file_predictions = []
    start_idx = 0
    for length in test_file_lengths:
        end_idx = start_idx + length
        predictions_for_this_file = y_pred[start_idx:end_idx]
        majority_vote = mode(predictions_for_this_file)[0]
        file_predictions.append([majority_vote, start_idx, end_idx])
        start_idx = end_idx

    first_column_list = [item[0] for item in file_predictions]

    #print(first_column_list)

    y_test_files = X_test_sort['Delirium'].values

    #print(type(y_test_files))

    y_test_files_copy = X_test_sort

    y_test_files_copy = y_test_files_copy.assign(Predictions=first_column_list)

    print("各テストファイルごとの分類結果\n")
    print(y_test_files_copy.drop(columns=['filepath'], errors='ignore'))

    correct_files = np.sum(np.array(first_column_list) == y_test_files)
    total_files = len(X_test_sort)
    file_accuracy = correct_files/ total_files

    scores = []
    scores.append(file_accuracy)
    print("\n")
    print(f"正解率 (ファイル単位): {file_accuracy:.4f} ({correct_files}/{total_files} ファイル正解)\n")


    #8.------混同行列の作成------
    cm = confusion_matrix(y_test_files, first_column_list)
    cm_df = pd.DataFrame(cm, 
                        index=['正解: 0', '正解: 1'], 
                        columns=['予測: 0', '予測: 1'])

    print("混同行列:")
    print(cm_df)
    print("\n") # 見やすいように改行

    #特徴量の重要度を取得
    feature_names = X_train_model.columns.tolist()
        
    importances = best_model.feature_importances_

    #特徴量の名前と重要度をまとめたDataFrameを作成
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })

    #重要度が高い順にソート
    sorted_importance_df = importance_df.sort_values(by='Importance', ascending=False)

    print(sorted_importance_df) """




ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required.

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import os
import re
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import mode
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
from boruta import BorutaPy

pd.options.display.max_rows = None

#表情データの読み込みとdfに格納
folder_path = Path('../../output_filtered_questions')
voice_path = Path('../voice/voice_csv_20ms')

file_list = folder_path.glob('*.csv')
voice_list = voice_path.glob('*.csv')

file_path_list = [str(p) for p in file_list]
voice_path_list = [str(p) for p in voice_list]

df = pd.DataFrame(file_path_list, columns=['filepath'])
voice_df = pd.DataFrame(voice_path_list, columns=['voice_filepath'])
#print(voice_df)


files_df = df
files_df['voice'] = voice_df['voice_filepath']
files_df['filename'] = df['filepath'].apply(lambda p: Path(p).name)
files_df['ID'] = files_df['filename'].str.extract(r'ID(\d+)')
files_df['ID'] = files_df['ID'].astype(int)

#患者の疾患有無
labels_df = pd.read_csv(r"C:\Users\robotics\proj\Research\code\voice\delirium.csv")
#患者と疾患有無を結合
file_with_labels_df = pd.merge(files_df, labels_df, on='ID', how='left')
file_with_labels_df.drop(columns=['Sex'], inplace=True)

print(file_with_labels_df)

#学習・テストデータとなるファイルの分割
X_files = file_with_labels_df.drop(columns='Delirum', errors='ignore')
#print(X_files)
y_labels = file_with_labels_df.drop(columns=['filepath', 'voice', 'filename', 'ID'], errors='ignore')
#rint(y_labels)

                                             filepath  \
0   ..\..\output_filtered_questions\filtered_ID36.csv   
1   ..\..\output_filtered_questions\filtered_ID37.csv   
2   ..\..\output_filtered_questions\filtered_ID38.csv   
3   ..\..\output_filtered_questions\filtered_ID39.csv   
4   ..\..\output_filtered_questions\filtered_ID40.csv   
5   ..\..\output_filtered_questions\filtered_ID41.csv   
6   ..\..\output_filtered_questions\filtered_ID42.csv   
7   ..\..\output_filtered_questions\filtered_ID43.csv   
8   ..\..\output_filtered_questions\filtered_ID44.csv   
9   ..\..\output_filtered_questions\filtered_ID45.csv   
10  ..\..\output_filtered_questions\filtered_ID46.csv   
11  ..\..\output_filtered_questions\filtered_ID47.csv   
12  ..\..\output_filtered_questions\filtered_ID48.csv   
13  ..\..\output_filtered_questions\filtered_ID49.csv   
14  ..\..\output_filtered_questions\filtered_ID50.csv   
15  ..\..\output_filtered_questions\filtered_ID51.csv   
16  ..\..\output_filtered_quest

In [9]:
#層化５分割交差検証
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_index, test_index) in enumerate(skf.split(X_files, y_labels)):
    X_train_sort, X_test_sort = X_files.iloc[train_index], X_files.iloc[test_index]
    y_train_sort, y_test_sort = y_labels.iloc[train_index], y_labels.iloc[test_index]

    #2.------学習データを読み込み、質問ごとの最大値を計算------

    train_file_path = [f for f in X_train_sort['filepath']]
    train_voice_path = [v for v in X_train_sort['voice']]
    #print(train_file_path)
    train_df = pd.DataFrame()
    for f, v, a in zip(train_file_path, train_voice_path, y_train_sort['Delirium']):
        df_train_file = pd.read_csv(f)
        df_train_voice = pd.read_csv(v)
        #print(df_train_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #ファイル名の数字部分を抽出
        number = re.findall(r'\d+', current_filename)
        #print(number)
        df_train_file['Delirium'] = a
        df_train_file['filenumber'] = number[0]
        #print(df_train_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_train_file['label'] = df_train_file['label'].astype(str).str.strip()
        df_train_voice['label'] = df_train_voice['label'].str.strip()
        
        #応答部分のAUだけを抽出
        df_train_sort_df = df_train_file[df_train_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_train_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_train_sort_df[confidence].index
        df_train_new_df = df_train_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_train_new_df.columns if col.startswith("AU")] + ["Delirium", "filenumber"]
        df_train_final_df = df_train_new_df[columns_to_extract]

        df_train_voice_df = df_train_voice[df_train_voice['label'].str.match('^answer(10|[1-9])$', na=False)]
        df_train_voice_drop = df_train_voice_df.dropna()

        #print(df_train_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]
        columns_to_voice = ["start", "end", "Loudness_sma3", "alphaRatio_sma3", "hammarbergIndex_sma3", "slope0-500_sma3", "slope500-1500_sma3", "logRelF0-H1-H2_sma3nz", "logRelF0-H1-A3_sma3nz", "original_start", "original_end"]
        # "mfcc1_sma3", "mfcc2_sma3", "mfcc3_sma3", "mfcc4_sma3",
        #AU07, 11, 20の削除
        df_train_final_df = df_train_final_df.drop(columns=columns_to_AU, errors='ignore')
        df_train_voice_data = df_train_voice_drop.drop(columns=columns_to_voice, errors='ignore')
        #print(df_train_final_df)
        #print(df_train_voice_data)

        #数値型に変更
        #df_train_final_df['filename'] = pd.to_numeric(df_train_final_df['filename'], errors='coerce')
        #print(df_train_final_df)

        #質問ごとの最大値を計算
        df_train_max_df = df_train_final_df.groupby('label').max()
        df_train_max_df = df_train_max_df.reset_index(drop=True)

        df_train_max_voice = df_train_voice_data.groupby('label').max()
        df_train_max_voice = df_train_max_voice.reset_index(drop=True)
        #print(df_train_max_df)
        #print(df_train_max_voice)
        df_train_multi_df = pd.concat([df_train_max_df, df_train_max_voice], axis=1)
        
        train_df = pd.concat([train_df, df_train_multi_df], ignore_index=True)

        train_df = train_df.dropna()

    #print(train_df.columns)

    #3.------テストデータを読み込み、質問ごとの最大値を計算------

    test_file_path = [f for f in X_test_sort['filepath']]
    test_voice_path = [v for v in X_test_sort['voice']]
    #print(test_file_path)
    test_df = pd.DataFrame()
    test_file_lengths = []
    for f, v, a in zip(test_file_path, test_voice_path, y_test_sort['Delirium']):
        df_test_file = pd.read_csv(f)
        df_test_voice = pd.read_csv(v)
        #print(df_test_file)
        #ファイル名とラベルの付与
        current_filename = os.path.basename(f)
        #print(current_filename)
        df_test_file['Delirium'] = a
        df_test_file['filename'] = current_filename
        #print(df_test_file)  
        
        #必要のない列の削除
        #label列にある空白の削除
        df_test_file['label'] = df_test_file['label'].astype(str).str.strip()
        df_test_voice['label'] = df_test_voice['label'].str.strip()
        
        #応答部分のAUだけを抽出
        df_test_sort_df = df_test_file[df_test_file['label'].str.match('^answer(10|[1-9])$', na=False)]
        confidence = df_test_sort_df['Confidence'] <= 0.90
        rows_to_delete = df_test_sort_df[confidence].index
        df_test_new_df = df_test_sort_df.drop(rows_to_delete)
        columns_to_extract = ["label"] + [col for col in df_test_new_df.columns if col.startswith("AU")] + ["Delirium", "filename"]
        df_test_final_df = df_test_new_df[columns_to_extract]

        df_test_voice_df = df_test_voice[df_test_voice['label'].str.match('^answer(10|[1-9])$', na=False)]
        df_test_voice_drop = df_test_voice_df.dropna()

        #print(df_test_final_df)

        columns_to_AU = ["AU07", "AU11", "AU20"]
        columns_to_voice = ["start", "end", "Loudness_sma3", "alphaRatio_sma3", "hammarbergIndex_sma3", "slope0-500_sma3", "slope500-1500_sma3",  "logRelF0-H1-H2_sma3nz", "logRelF0-H1-A3_sma3nz", "original_start", "original_end"]
        #"mfcc1_sma3", "mfcc2_sma3", "mfcc3_sma3", "mfcc4_sma3",
        #AU07, 11, 20の削除
        df_test_final_df = df_test_final_df.drop(columns=columns_to_AU, errors='ignore')
        df_test_voice_data = df_test_voice_drop.drop(columns=columns_to_voice, errors='ignore')
        #print(df_test_final_df)

        #数値型に変更
        #df_test_final_df['filename'] = pd.to_numeric(df_test_final_df['filename'], errors='coerce')
        #print(df_test_final_df)

        #質問ごとの最大値を計算
        df_test_max_df = df_test_final_df.groupby('label').max()
        df_test_max_df = df_test_max_df.reset_index(drop=True)

        df_test_max_voice = df_test_voice_data.groupby('label').mean()
        df_test_max_voice = df_test_max_voice.reset_index(drop=True)
        #print(df_test_max_df)
        #print(df_test_max_voice)
        
        test_file_lengths.append(len(df_test_max_df))

        df_test_multi_df = pd.concat([df_test_max_df, df_test_max_voice], axis=1)

        test_df = pd.concat([test_df, df_test_multi_df], ignore_index=True)

        test_df = test_df.dropna()

    #print(test_df)
    #print(test_file_lengths)

    #4.------グリッドサーチのための学習・検証データの用意とテストデータの用意------

    y_train_grid = train_df['Delirium']
    y_test_grid = test_df['Delirium']

    columns_to_delete = ['Delirium']
    columns_to_delete2 = ['Delirium', 'filename']

    X_train_grid = train_df.drop(columns=columns_to_delete, errors='ignore')
    X_test_grid = test_df.drop(columns=columns_to_delete2, errors='ignore')

    #print(y_train_grid)

    #患者ごとに分割するための変数を用意し、学習データからfilenumberを削除
    patient_groups = X_train_grid['filenumber'].values
    X_train_boruta = X_train_grid.drop(columns=['filenumber']).copy()

    #Add.------学習データだけを使用してBorutaを実行------

    #Borutaの実行
    Rf_boruta = RandomForestClassifier(n_jobs=-1, class_weight='balanced', max_depth=5)
    feat_selector = BorutaPy(Rf_boruta, n_estimators='auto', verbose=2, random_state=1, perc=80)
    feat_selector.fit(X_train_boruta, y_train_grid)
    X_train_model = X_train_boruta.loc[:, feat_selector.support_]
    X_test_model = X_test_grid.loc[:, feat_selector.support_]

    #Borutaで採用、削除した特徴量の確認
    feature_names = np.array(X_train_boruta.columns)
    dropped_features = feature_names[~feat_selector.support_]
    accepted_features = feature_names[feat_selector.support_]
    tentative_features = feature_names[feat_selector.support_weak_]

    print(f"削除前の特徴量数：{len(feature_names)}")
    print(f"削除された特徴量数：{len(dropped_features)}")
    print("削除された特徴量一覧")
    print(dropped_features)

    #5.------学習の準備(学習器と学習方法の設定)------

    #モデル・グリッドサーチの定義
    model = RandomForestClassifier(random_state=42)
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 5, 10, 15],
        'min_samples_split': [2, 5, 10]
        }
    grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=cv, n_jobs=-1)

    #6.------グリッドサーチ------
    grid_search.fit(X_train_model, y_train_grid, groups=patient_groups)
    print(f"---fold{fold+1}---\n")
    print(f"最適なハイパーパラメータ：{grid_search.best_params_}")
    print(f"CVでのmax精度(Accuracy):{grid_search.best_score_:.4f}")

    #行と列の表示制限を解除（または十分に大きな値に設定）
    pd.set_option('display.max_rows', None)     # すべての行を表示
    pd.set_option('display.max_columns', None)  # すべての列を表示
    pd.set_option('display.width', 1000)        # 表示幅を広くする

    #結果をDataFrameに変換し表示
    results_df = pd.DataFrame(grid_search.cv_results_)

    print("--- グリッドサーチの全試行結果 ---")
    print(results_df)

    #表示設定を元に戻す (推奨)
    #グローバルな設定を元に戻し、他の処理に影響を与えないようにします
    pd.reset_option('display.max_rows')
    pd.reset_option('display.max_columns')
    pd.reset_option('display.width')


    #7.------テストデータでの性能評価------
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test_model)
    final_accuracy = accuracy_score(y_test_grid, y_pred)

    print(f"テストデータでの正解率: {final_accuracy:.4}\n")
    #print(y_pred)

    file_predictions = []
    start_idx = 0
    for length in test_file_lengths:
        end_idx = start_idx + length
        predictions_for_this_file = y_pred[start_idx:end_idx]
        majority_vote = mode(predictions_for_this_file)[0]
        file_predictions.append([majority_vote, start_idx, end_idx])
        start_idx = end_idx

    first_column_list = [item[0] for item in file_predictions]

    #print(first_column_list)

    y_test_files = X_test_sort['Delirium'].values

    #print(type(y_test_files))

    y_test_files_copy = X_test_sort

    y_test_files_copy = y_test_files_copy.assign(Predictions=first_column_list)

    print("各テストファイルごとの分類結果\n")
    print(y_test_files_copy.drop(columns=['filepath'], errors='ignore'))

    correct_files = np.sum(np.array(first_column_list) == y_test_files)
    total_files = len(X_test_sort)
    file_accuracy = correct_files/ total_files

    scores = []
    scores.append(file_accuracy)
    print("\n")
    print(f"正解率 (ファイル単位): {file_accuracy:.4f} ({correct_files}/{total_files} ファイル正解)\n")


    #8.------混同行列の作成------
    cm = confusion_matrix(y_test_files, first_column_list)
    cm_df = pd.DataFrame(cm, 
                        index=['正解: 0', '正解: 1'], 
                        columns=['予測: 0', '予測: 1'])

    print("混同行列:")
    print(cm_df)
    print("\n") # 見やすいように改行

    #特徴量の重要度を取得 """
    feature_names = X_train_model.columns.tolist()
        
    importances = best_model.feature_importances_

    #特徴量の名前と重要度をまとめたDataFrameを作成
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    })

    #重要度が高い順にソート
    sorted_importance_df = importance_df.sort_values(by='Importance', ascending=False)

    print(sorted_importance_df)

Iteration: 	1 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	2 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	3 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	4 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	5 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	6 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	7 / 100
Confirmed: 	0
Tentative: 	35
Rejected: 	0
Iteration: 	8 / 100
Confirmed: 	13
Tentative: 	17
Rejected: 	5
Iteration: 	9 / 100
Confirmed: 	13
Tentative: 	17
Rejected: 	5
Iteration: 	10 / 100
Confirmed: 	13
Tentative: 	17
Rejected: 	5
Iteration: 	11 / 100
Confirmed: 	13
Tentative: 	17
Rejected: 	5
Iteration: 	12 / 100
Confirmed: 	16
Tentative: 	12
Rejected: 	7
Iteration: 	13 / 100
Confirmed: 	16
Tentative: 	12
Rejected: 	7
Iteration: 	14 / 100
Confirmed: 	16
Tentative: 	12
Rejected: 	7
Iteration: 	15 / 100
Confirmed: 	16
Tentative: 	12
Rejected: 	7
Iteration: 	16 / 100
Confirmed: 	16
Tentative: 	12
Rejec